In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dataset/Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv")

print(df.shape)
print(df.head())

(8173, 46)
           id event_type   latitude  longitude  endlatitude  endlongitude  \
0  FKID000000  unplanned  13.040004  77.518099     0.000000      0.000000   
1  FKID000001  unplanned  12.921876  77.645158     0.000000      0.000000   
2  FKID000002  unplanned  12.955622  77.585708     0.000000      0.000000   
3  FKID000003  unplanned  13.006147  77.579435    13.006239     77.579516   
4  FKID000004  unplanned  12.953980  77.585233     0.000000      0.000000   

                                             address  \
0  Mumbai Bengaluru Highway, Jalahalli Cross Junc...   
1  19th Main Road, Heavie Halcyon, Agara, HSR Lay...   
2  Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...   
3  Sankey Road, Bashyam Circle, Sadashiva Nagar, ...   
4  Lalbagh Fort Road, Lalbagh Main Gate Junction,...   

                                         end_address        event_cause  \
0                                                NaN  vehicle_breakdown   
1                                      

In [16]:
df = df[
    [
        "event_type",
        "event_cause",
        "priority",
        "corridor",
        "police_station",
        "latitude",
        "longitude",
        "requires_road_closure",
        "start_datetime"
    ]
]

In [17]:
df["start_datetime"] = pd.to_datetime(
    df["start_datetime"],
    format="mixed",
    errors="coerce",
    utc=True
)

df["hour"] = df["start_datetime"].dt.hour
df["day"] = df["start_datetime"].dt.dayofweek
df["month"] = df["start_datetime"].dt.month

In [18]:
print(df.columns.tolist())

['event_type', 'event_cause', 'priority', 'corridor', 'police_station', 'latitude', 'longitude', 'requires_road_closure', 'start_datetime', 'hour', 'day', 'month']


In [19]:
for col in df.columns:
    print(repr(col))

'event_type'
'event_cause'
'priority'
'corridor'
'police_station'
'latitude'
'longitude'
'requires_road_closure'
'start_datetime'
'hour'
'day'
'month'


In [20]:
print(df.shape)
print(df.head())

print("\nHour column:")
print(df["hour"].head())

print("\nMissing values in hour:")
print(df["hour"].isna().sum())

(8173, 12)
  event_type        event_cause priority      corridor  police_station  \
0  unplanned  vehicle_breakdown     High   Tumkur Road          Peenya   
1  unplanned  vehicle_breakdown     High    ORR East 1      HSR Layout   
2  unplanned             others      Low  Non-corridor   Wilson Garden   
3  unplanned          tree_fall      Low  Non-corridor  Sadashivanagar   
4  unplanned  vehicle_breakdown      Low  Non-corridor   Wilson Garden   

    latitude  longitude  requires_road_closure  \
0  13.040004  77.518099                  False   
1  12.921876  77.645158                  False   
2  12.955622  77.585708                  False   
3  13.006147  77.579435                   True   
4  12.953980  77.585233                  False   

                    start_datetime  hour  day  month  
0 2024-03-07 17:01:48.111000+00:00    17    3      3  
1 2024-01-30 04:07:24.173000+00:00     4    1      1  
2 2023-11-11 06:18:03.343000+00:00     6    5     11  
3 2024-03-07 17:56:55.0

In [21]:
print(df[["start_datetime", "hour"]].head())

                    start_datetime  hour
0 2024-03-07 17:01:48.111000+00:00    17
1 2024-01-30 04:07:24.173000+00:00     4
2 2023-11-11 06:18:03.343000+00:00     6
3 2024-03-07 17:56:55.061000+00:00    17
4 2024-01-30 04:56:32.348000+00:00     4


In [22]:
def create_severity(row):

    score = 0

    # Priority

    if row["priority"] == "High":
        score += 3

    # Road closure

    if row["requires_road_closure"]:
        score += 3

    # Event type

    if row["event_type"] == "unplanned":
        score += 2

    # Event cause

    critical_causes = [
        "accident",
        "water_logging",
        "construction",
        "public_event",
        "procession",
        "vip_movement",
        "protest"
    ]

    if row["event_cause"] in critical_causes:
        score += 2

    # Peak hours

    if row["hour"] in [5,6,19,20,21]:
        score += 1

    # Zone

    hot_zones = [
        "Central Zone 2",
        "West Zone 1",
        "North Zone 2"
    ]

    if row["zone"] in hot_zones:
        score += 1

    if score >= 8:
        return "Critical"

    elif score >= 6:
        return "High"

    elif score >= 3:
        return "Medium"

    else:
        return "Low"

In [23]:
print(df["severity"].value_counts())

KeyError: 'severity'

In [24]:
df["severity"] = df.apply(
    create_severity,
    axis=1
)

KeyError: 'zone'

In [33]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

cat_cols = [
    "event_type",
    "event_cause",
    "priority",
    "corridor",
    "police_station"
]

for col in cat_cols:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    encoders[col] = le

In [19]:
X = df[
    [
        "event_type",
        "event_cause",
        "priority",
        "corridor",
        "police_station",
        "latitude",
        "longitude",
        "hour",
        "day"
    ]
]

y = df["severity"]

In [20]:
severity_encoder = LabelEncoder()

y = severity_encoder.fit_transform(y)

In [21]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [22]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [23]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)

print(
    accuracy_score(y_test,pred)
)

0.9284403669724771


In [24]:
import joblib

joblib.dump(
    model,
    "../models/severity_model.pkl"
)

joblib.dump(
    encoders,
    "../models/encoders.pkl"
)

joblib.dump(
    severity_encoder,
    "../models/severity_encoder.pkl"
)

['../models/severity_encoder.pkl']

In [25]:
def manpower(sev):

    if sev == "Critical":
        return 10

    elif sev == "High":
        return 6

    elif sev == "Medium":
        return 3

    else:
        return 1

df["manpower_required"] = df["severity"].apply(
    manpower
)

In [26]:
from xgboost import XGBRegressor

model = XGBRegressor()

model.fit(X, df["manpower_required"])

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [27]:
def diversion(row):

    if (
        row["severity"] == "Critical"
        or row["requires_road_closure"]
    ):
        return 1

    return 0


df["diversion_needed"] = df.apply(
    diversion,
    axis=1
)

In [28]:
from xgboost import XGBClassifier

div_model = XGBClassifier()

div_model.fit(
    X,
    df["diversion_needed"]
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [29]:
joblib.dump(
    div_model,
    "../models/diversion_model.pkl"
)

['../models/diversion_model.pkl']

In [1]:
import pandas as pd

df = pd.read_csv(
    "../dataset/severity_dataset.csv"
)

In [3]:
features = [

    "event_type",

    "event_cause",

    "priority",

    "requires_road_closure",

    "zone",

    "police_station",

    "latitude",

    "longitude",

    "hour",

    "day"
]

In [4]:
from sklearn.preprocessing import LabelEncoder

encoders = {}

for col in [

    "event_type",
    "event_cause",
    "priority",
    "zone",
    "police_station"

]:

    le = LabelEncoder()

    df[col] = le.fit_transform(
        df[col].astype(str)
    )

    encoders[col] = le

In [5]:
target_encoder = LabelEncoder()

df["severity"] = target_encoder.fit_transform(
    df["severity"]
)

In [6]:
from sklearn.model_selection import train_test_split

X = df[features]

y = df["severity"]

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
from sklearn.preprocessing import LabelEncoder

cat_cols = X_train.select_dtypes(include=["object"]).columns

for col in cat_cols:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

In [9]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [10]:
from sklearn.metrics import classification_report

pred = model.predict(X_test)

print(
    classification_report(
        y_test,
        pred
    )
)

              precision    recall  f1-score   support

           0       0.96      0.96      0.96       108
           1       0.99      0.98      0.98       563
           2       1.00      1.00      1.00       231
           3       0.99      1.00      0.99       733

    accuracy                           0.99      1635
   macro avg       0.98      0.98      0.98      1635
weighted avg       0.99      0.99      0.99      1635



In [11]:
import joblib

joblib.dump(
    model,
    "../models/severity_model.pkl"
)

joblib.dump(
    encoders,
    "../models/encoders.pkl"
)

joblib.dump(
    target_encoder,
    "../models/severity_encoder.pkl"
)

['../models/severity_encoder.pkl']

In [12]:
def congestion_score(row):

    score = 0

    if row["priority"] == "High":
        score += 25

    if row["requires_road_closure"]:
        score += 25

    if row["event_type"] == "unplanned":
        score += 15

    if row["event_cause"] in [
        "accident",
        "construction",
        "water_logging"
    ]:
        score += 20

    if row["hour"] in [5,6,19,20,21]:
        score += 15

    return min(score,100)

In [13]:
df["congestion_score"] = df.apply(
    congestion_score,
    axis=1
)

In [14]:
df.to_csv(
    "../dataset/congestion_dataset.csv",
    index=False
)